# Generate Salesforce Sample Data

This notebook generates reproducible, Salesforce-style sample CSV files and writes them to the Unity Catalog volume given by the `volume_base` parameter (dev: `/Volumes/ofer_demo_dev/bronze/salesforce`, prod: `/Volumes/ofer_demo_prod/bronze/salesforce`):

* `accounts/accounts.csv` - ~20 accounts
* `transactions/transactions.csv` - 3 to 8 transactions per account
* `contacts/contacts.csv` - 2 to 5 customer contact records per account (loaded to `sf_contacts`)
* `opportunities/opportunities.csv` - 1 to 4 sales opportunities per account
* `tickets/tickets.csv` - 3 to 12 Upwind support tickets per account (bugs, incidents, SLA, CSAT/NPS feedback), plus 12-18 extra recent tickets each for Stark Labs, Globex Labs and Blue Sun Labs

These five files are the **baseline**. A fixed random seed (42) always produces the same baseline, and it is written only if the files do not exist yet (Step 5, `REGENERATE_BASE = False`), so re-running never rewrites it.

The notebook can be run any number of times, for example from a scheduled job. **Every run adds a new batch** (Step 5b): one new timestamped file per entity, such as `accounts/accounts_20261103T213000Z.csv`. Each batch holds 2-5 new accounts with their transactions, contacts, opportunities and tickets, dated relative to the real current date, with IDs continuing from the highest existing ID. The accounts file also contains 1-2 invalid account rows to exercise the pipeline's `sf_accounts` data quality checks. All other files in a batch are fully valid.

## Step 1: Configuration
Set the volume path, random seed, number of accounts, and the reference value lists used to build realistic records.

In [0]:
import csv
import os
import random
from datetime import date, timedelta

dbutils.widgets.text("volume_base", "/Volumes/ofer_demo_dev/bronze/salesforce")
VOLUME_PATH = dbutils.widgets.get("volume_base")
SEED = 42
NUM_ACCOUNTS = 20
rng = random.Random(SEED)

COMPANY_PREFIXES = ["Acme", "Globex", "Initech", "Umbrella", "Stark", "Wayne", "Wonka", "Cyberdyne", "Soylent", "Hooli",
                    "Vandelay", "Pied Piper", "Tyrell", "Oscorp", "Massive Dynamic", "Aperture", "Blue Sun", "Gringotts",
                    "Monarch", "Nakatomi", "Dunder", "Prestige"]
COMPANY_SUFFIXES = ["Inc", "Corp", "LLC", "Group", "Holdings", "Technologies", "Systems", "Labs"]
INDUSTRIES = ["Technology", "Financial Services", "Healthcare", "Retail", "Manufacturing", "Media", "Energy", "Telecommunications"]
ACCOUNT_TYPES = ["Customer", "Partner", "Prospect"]
COUNTRIES = ["United States", "United Kingdom", "Germany", "France", "Israel", "Canada", "Australia", "Japan"]
COUNTRY_CURRENCY = {"United States": "USD", "United Kingdom": "GBP", "Germany": "EUR", "France": "EUR",
                    "Israel": "USD", "Canada": "CAD", "Australia": "AUD", "Japan": "JPY"}
PRODUCT_TYPES = ["Subscription", "License", "Services", "Support"]
STATUSES = ["Completed", "Pending", "Refunded"]
STATUS_WEIGHTS = [0.8, 0.15, 0.05]
FIRST_NAMES = ["James", "Mary", "John", "Patricia", "Robert", "Jennifer", "Michael", "Linda", "David", "Sarah",
               "Daniel", "Noa", "Yael", "Omer", "Emma", "Liam", "Olivia", "Lucas", "Sofia", "Ethan"]
LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Cohen", "Levi",
              "Martin", "Lee", "Walker", "Hall", "Young", "King", "Wright", "Lopez", "Hill", "Scott"]
TITLES_BY_DEPT = {
    "Engineering": ["Software Engineer", "Engineering Manager", "VP Engineering", "CTO"],
    "Finance": ["Financial Analyst", "Controller", "CFO"],
    "Sales": ["Account Executive", "Sales Manager", "VP Sales"],
    "Marketing": ["Marketing Manager", "CMO"],
    "IT": ["IT Manager", "Systems Administrator", "CIO"],
    "Operations": ["Operations Manager", "COO"],
    "Procurement": ["Procurement Specialist", "Head of Procurement"],
}
TODAY = date(2026, 10, 30)


def random_date(start: date, end: date) -> date:
    return start + timedelta(days=rng.randint(0, max((end - start).days, 0)))


def write_csv(subfolder: str, filename: str, rows: list, columns: list) -> str:
    """Write rows to <VOLUME_PATH>/<subfolder>/<filename> with a header, overwriting existing files."""
    folder = f"{VOLUME_PATH}/{subfolder}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{filename}"
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    return path

## Step 2: Generate accounts
Create ~20 accounts with sequential IDs (`ACC00001`...), unique company names, firmographics (industry, type, country, size, revenue), a website derived from the name, and a `created_date` between 2018 and 2024.

In [0]:
ACCOUNT_COLUMNS = ["account_id", "account_name", "industry", "account_type", "country",
                   "employee_count", "annual_revenue", "website", "created_date"]

accounts = []
prefixes = rng.sample(COMPANY_PREFIXES, NUM_ACCOUNTS)
for i, prefix in enumerate(prefixes, start=1):
    suffix = rng.choice(COMPANY_SUFFIXES)
    employee_count = rng.choice([rng.randint(10, 200), rng.randint(200, 2000), rng.randint(2000, 50000)])
    revenue_per_employee = rng.uniform(80_000, 400_000)
    domain = prefix.lower().replace(" ", "")
    accounts.append({
        "account_id": f"ACC{i:05d}",
        "account_name": f"{prefix} {suffix}",
        "industry": rng.choice(INDUSTRIES),
        "account_type": rng.choices(ACCOUNT_TYPES, weights=[0.6, 0.2, 0.2])[0],
        "country": rng.choice(COUNTRIES),
        "employee_count": employee_count,
        "annual_revenue": round(employee_count * revenue_per_employee, -3),
        "website": f"https://www.{domain}.com",
        "created_date": random_date(date(2018, 1, 1), date(2024, 12, 31)).isoformat(),
    })

print(f"Generated {len(accounts)} accounts")
accounts[:3]

## Step 3: Generate transactions
For each account create 3 to 8 transactions. Each `transaction_date` falls strictly after the account's `created_date`, the currency follows the account's country, and amounts scale with product type. Status is weighted toward `Completed`.

In [0]:
TRANSACTION_COLUMNS = ["transaction_id", "account_id", "transaction_date", "amount",
                       "currency", "product_type", "status"]
AMOUNT_RANGES = {"Subscription": (1_000, 50_000), "License": (5_000, 250_000),
                 "Services": (2_000, 100_000), "Support": (500, 20_000)}

transactions = []
txn_seq = 1
for acc in accounts:
    created = date.fromisoformat(acc["created_date"])
    for _ in range(rng.randint(3, 8)):
        product = rng.choice(PRODUCT_TYPES)
        low, high = AMOUNT_RANGES[product]
        transactions.append({
            "transaction_id": f"TXN{txn_seq:06d}",
            "account_id": acc["account_id"],
            "transaction_date": random_date(created + timedelta(days=1), TODAY).isoformat(),
            "amount": round(rng.uniform(low, high), 2),
            "currency": COUNTRY_CURRENCY[acc["country"]],
            "product_type": product,
            "status": rng.choices(STATUSES, weights=STATUS_WEIGHTS)[0],
        })
        txn_seq += 1

print(f"Generated {len(transactions)} transactions")
transactions[:3]

## Step 4: Generate employees / contacts
For each account create 2 to 5 contact records with a department-appropriate title, an email on the account's domain, a phone number, and a `start_date` on or after the account's `created_date`.

In [0]:
EMPLOYEE_COLUMNS = ["contact_id", "account_id", "first_name", "last_name", "email",
                    "title", "department", "phone", "start_date"]

employees = []
emp_seq = 1
for acc in accounts:
    domain = acc["website"].replace("https://www.", "")
    created = date.fromisoformat(acc["created_date"])
    used_names = set()
    for _ in range(rng.randint(2, 5)):
        while True:
            first, last = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
            if (first, last) not in used_names:
                used_names.add((first, last))
                break
        dept = rng.choice(list(TITLES_BY_DEPT))
        employees.append({
            "contact_id": f"EMP{emp_seq:06d}",
            "account_id": acc["account_id"],
            "first_name": first,
            "last_name": last,
            "email": f"{first.lower()}.{last.lower()}@{domain}",
            "title": rng.choice(TITLES_BY_DEPT[dept]),
            "department": dept,
            "phone": f"+1-{rng.randint(200, 999)}-{rng.randint(200, 999)}-{rng.randint(1000, 9999)}",
            "start_date": random_date(created, TODAY).isoformat(),
        })
        emp_seq += 1

print(f"Generated {len(employees)} employees")
employees[:3]

## Step 4b: Generate opportunities
For each account create 1 to 4 sales opportunities. Each one has a stage with a matching win probability, a type (New Business / Expansion / Renewal), a lead source, an amount in the account's currency, and a `primary_contact_id` that points to one of the account's own employees.

* `created_date` is on or after the account's `created_date`.
* Closed stages (`Closed Won` / `Closed Lost`) have a `close_date` in the past; open stages have an expected `close_date` in the next 9 months.

This cell runs after the employees step, so the seeded random sequence (and therefore accounts, transactions and employees) stays identical to previous runs.

In [0]:
OPPORTUNITY_COLUMNS = ["opportunity_id", "account_id", "opportunity_name", "type", "stage", "probability",
                       "amount", "currency", "lead_source", "primary_contact_id", "created_date", "close_date", "is_closed", "is_won"]
STAGE_PROBABILITY = {"Prospecting": 10, "Qualification": 20, "Needs Analysis": 40, "Proposal": 60,
                     "Negotiation": 80, "Closed Won": 100, "Closed Lost": 0}
STAGES = list(STAGE_PROBABILITY)
STAGE_WEIGHTS = [0.10, 0.12, 0.12, 0.14, 0.12, 0.25, 0.15]
OPP_TYPES = ["New Business", "Expansion", "Renewal"]
LEAD_SOURCES = ["Web", "Partner Referral", "Event", "Outbound", "Inbound"]
OPP_PRODUCTS = ["Cloud Security Platform", "CNAPP", "Runtime Protection", "Vulnerability Management", "Compliance Suite"]

employees_by_account = {}
for e in employees:
    employees_by_account.setdefault(e["account_id"], []).append(e["contact_id"])

opportunities = []
opp_seq = 1
for acc in accounts:
    created = date.fromisoformat(acc["created_date"])
    for _ in range(rng.randint(1, 4)):
        stage = rng.choices(STAGES, weights=STAGE_WEIGHTS)[0]
        is_closed = stage.startswith("Closed")
        opp_created = random_date(created, TODAY - timedelta(days=30))
        if is_closed:
            close = random_date(opp_created + timedelta(days=14), TODAY - timedelta(days=1))
        else:
            close = random_date(TODAY + timedelta(days=1), TODAY + timedelta(days=270))
        opp_type = rng.choice(OPP_TYPES)
        opportunities.append({
            "opportunity_id": f"OPP{opp_seq:06d}",
            "account_id": acc["account_id"],
            "opportunity_name": f"{acc['account_name']} - {rng.choice(OPP_PRODUCTS)} ({opp_type})",
            "type": opp_type,
            "stage": stage,
            "probability": STAGE_PROBABILITY[stage],
            "amount": round(rng.uniform(10_000, 500_000), -2),
            "currency": COUNTRY_CURRENCY[acc["country"]],
            "lead_source": rng.choice(LEAD_SOURCES),
            "primary_contact_id": rng.choice(employees_by_account[acc["account_id"]]),
            "created_date": opp_created.isoformat(),
            "close_date": close.isoformat(),
            "is_closed": is_closed,
            "is_won": stage == "Closed Won",
        })
        opp_seq += 1

print(f"Generated {len(opportunities)} opportunities")
opportunities[:3]

## Step 4c: Generate support tickets (bugs, incidents, customer feedback)
For each account create 3 to 12 support tickets about the **Upwind** cloud security platform: Runtime Sensor, CSPM, Vulnerability Management, KSPM, Cloud Detection & Response, Compliance, API Security, Integrations and Console. Each ticket has:

* **Lifecycle:** `created_at`, `first_response_at`, `resolved_at`, `status`, `reopen_count`, `escalated`, `support_tier`
* **Classification:** `priority` (P1-P4), `ticket_type` (Bug / Incident / Configuration / Question / Feature Request), `product_module`, `subject`, `cloud_provider`, `environment`, `channel`, `product_version`, `affected_assets`
* **Bugs:** `root_cause`, `is_bug`, `bug_id` (UPW-xxxx), `bug_severity`
* **SLA:** `sla_first_response_minutes`, `first_response_minutes`, `sla_breached`, `resolution_hours`
* **Customer feedback** (only on resolved/closed tickets that got a survey response): `csat_score` (1-5), `nps_score` (0-10), `sentiment`, `feedback_comment`

CSAT is driven by the ticket's experience: SLA breaches, reopens, long resolutions and P1 impact lower it. Tickets are created from May 2025 up to 3 days before today; recent tickets that are not resolved yet stay open.

This cell runs after opportunities, so all previously generated data stays identical.

In [0]:
from datetime import datetime

TICKET_COLUMNS = [
    "ticket_id", "account_id", "reporter_contact_id", "created_at", "first_response_at", "resolved_at",
    "status", "priority", "ticket_type", "product_module", "subject", "cloud_provider", "environment", "channel",
    "product_version", "affected_assets", "root_cause", "is_bug", "bug_id", "bug_severity", "support_tier",
    "escalated", "reopen_count", "sla_first_response_minutes", "first_response_minutes", "sla_breached",
    "resolution_hours", "csat_score", "nps_score", "sentiment", "feedback_comment",
]

# Upwind product modules -> realistic ticket subjects and their typical ticket type
TICKET_CATALOG = {
    "Runtime Sensor": [
        ("Sensor DaemonSet crashlooping after EKS 1.30 upgrade", "Bug"),
        ("High CPU usage from runtime sensor on large node pool", "Incident"),
        ("Sensor not reporting from GKE Autopilot nodes", "Bug"),
        ("eBPF sensor fails to load on Bottlerocket kernel", "Bug"),
        ("How to deploy the sensor with Helm behind an HTTP proxy", "Question"),
    ],
    "CSPM": [
        ("False positive: S3 bucket flagged public despite Block Public Access", "Bug"),
        ("Misconfiguration findings not updating after remediation", "Bug"),
        ("Azure subscription onboarding fails with insufficient permissions", "Configuration"),
        ("Request custom policy for mandatory resource tagging", "Feature Request"),
    ],
    "Vulnerability Management": [
        ("CVE count mismatch between console and SBOM export", "Bug"),
        ("Container image scan stuck in pending state", "Incident"),
        ("Missing vulnerabilities for distroless images", "Bug"),
        ("How is runtime reachability used to prioritize CVEs?", "Question"),
        ("Registry scanning for private ECR repositories not running", "Configuration"),
    ],
    "KSPM": [
        ("CIS Kubernetes benchmark report shows stale results", "Bug"),
        ("Admission controller blocking legitimate deployments", "Incident"),
        ("RBAC risk analysis missing ClusterRoleBindings", "Bug"),
    ],
    "Cloud Detection & Response": [
        ("Duplicate threat alerts for the same process execution", "Bug"),
        ("Delayed CloudTrail ingestion causing alert lag", "Incident"),
        ("Tune crypto-mining detection rule for CI runners", "Configuration"),
        ("Suspicious reverse shell alert - need investigation help", "Incident"),
        ("Add MITRE ATT&CK mapping to alert export", "Feature Request"),
    ],
    "Compliance & Reporting": [
        ("SOC 2 report export fails for large environments", "Bug"),
        ("PCI DSS framework mapping missing controls", "Bug"),
        ("Schedule weekly compliance PDF to executives", "Feature Request"),
    ],
    "API Security": [
        ("API endpoint discovery missing internal services", "Bug"),
        ("Shadow API detections show the wrong host", "Bug"),
        ("How to exclude health-check endpoints from API inventory", "Question"),
    ],
    "Integrations": [
        ("Jira integration not creating tickets for critical findings", "Bug"),
        ("Slack notifications delayed by several hours", "Incident"),
        ("Splunk HEC forwarding returns 403", "Configuration"),
        ("Terraform provider fails to onboard new cloud account", "Bug"),
        ("Request ServiceNow bi-directional sync", "Feature Request"),
    ],
    "Console & Access": [
        ("Dashboard widgets not loading for large organization", "Bug"),
        ("SSO login loop with Okta", "Incident"),
        ("Custom RBAC role cannot view findings", "Configuration"),
        ("Add dark mode to the console", "Feature Request"),
    ],
}
MODULES = list(TICKET_CATALOG)
MODULE_WEIGHTS = [0.20, 0.14, 0.15, 0.09, 0.14, 0.07, 0.06, 0.09, 0.06]

PRIORITIES = ["P1 - Critical", "P2 - High", "P3 - Medium", "P4 - Low"]
PRIORITY_WEIGHTS_BY_TYPE = {
    "Incident": [0.35, 0.40, 0.20, 0.05],
    "Bug": [0.12, 0.35, 0.38, 0.15],
    "Configuration": [0.03, 0.22, 0.50, 0.25],
    "Question": [0.00, 0.08, 0.47, 0.45],
    "Feature Request": [0.00, 0.05, 0.35, 0.60],
}
SLA_FIRST_RESPONSE_MIN = {"P1 - Critical": 30, "P2 - High": 120, "P3 - Medium": 480, "P4 - Low": 1440}
FIRST_RESPONSE_RANGE_MIN = {"P1 - Critical": (3, 70), "P2 - High": (10, 220), "P3 - Medium": (30, 800), "P4 - Low": (60, 2300)}
RESOLUTION_RANGE_H = {"P1 - Critical": (2, 48), "P2 - High": (6, 120), "P3 - Medium": (24, 336), "P4 - Low": (48, 720)}
BUG_SEVERITY = {"P1 - Critical": "S1", "P2 - High": "S2", "P3 - Medium": "S3", "P4 - Low": "S4"}
ROOT_CAUSES_BY_TYPE = {
    "Bug": (["Product Bug", "Product Bug", "Product Bug", "Third-party Integration", "Customer Environment"]),
    "Incident": (["Product Bug", "Cloud Provider Issue", "Customer Environment", "Third-party Integration", "Capacity / Scaling"]),
    "Configuration": (["Configuration", "Configuration", "Customer Environment", "Documentation Gap"]),
    "Question": (["Documentation Gap", "Not a Bug", "Not a Bug"]),
    "Feature Request": (["Feature Gap"]),
}
OPEN_STATUSES = ["New", "In Progress", "Waiting on Customer"]
CLOUDS, CLOUD_WEIGHTS = ["AWS", "Azure", "GCP", "Multi-cloud"], [0.50, 0.20, 0.20, 0.10]
ENVIRONMENTS, ENV_WEIGHTS = ["Production", "Staging", "Development"], [0.65, 0.25, 0.10]
CHANNELS, CHANNEL_WEIGHTS = ["Portal", "Email", "Slack Connect", "In-app", "Phone"], [0.35, 0.25, 0.25, 0.10, 0.05]
FEEDBACK = {
    "Positive": [
        "Support resolved our {m} issue quickly and explained the root cause clearly.",
        "Great response time from the Upwind team on the {m} problem.",
        "Very knowledgeable engineer, the {m} fix worked on the first try.",
        "Appreciated the proactive updates while the {m} issue was being fixed.",
    ],
    "Neutral": [
        "The {m} issue was fixed, but it took a few back-and-forths.",
        "Resolution was fine, documentation for {m} could be clearer.",
        "OK experience overall; would like faster first response on {m} tickets.",
    ],
    "Negative": [
        "Too slow to resolve our {m} problem; it impacted production security coverage.",
        "The {m} issue came back after it was marked resolved.",
        "Had to escalate twice before anyone looked at the {m} ticket.",
        "Missed the SLA on a critical {m} incident - very disappointing.",
    ],
}

TICKETS_START = datetime(2025, 5, 1)
NOW = datetime(TODAY.year, TODAY.month, TODAY.day, 9, 0)
TICKETS_END = NOW - timedelta(days=3)

def fmt_ts(ts):
    return ts.strftime("%Y-%m-%d %H:%M:%S") if ts else None

tickets = []
tkt_seq = 1
for acc in accounts:
    acc_created = datetime.fromisoformat(acc["created_date"])
    window_start = max(acc_created, TICKETS_START)
    window_seconds = int((TICKETS_END - window_start).total_seconds())
    contacts = employees_by_account[acc["account_id"]]
    for _ in range(rng.randint(3, 12)):
        module = rng.choices(MODULES, weights=MODULE_WEIGHTS)[0]
        subject, ttype = rng.choice(TICKET_CATALOG[module])
        priority = rng.choices(PRIORITIES, weights=PRIORITY_WEIGHTS_BY_TYPE[ttype])[0]
        created_at = window_start + timedelta(seconds=rng.randint(0, window_seconds))

        sla_min = SLA_FIRST_RESPONSE_MIN[priority]
        lo, hi = FIRST_RESPONSE_RANGE_MIN[priority]
        first_resp_min = rng.randint(lo, hi)
        sla_breached = first_resp_min > sla_min
        first_response_at = created_at + timedelta(minutes=first_resp_min)

        lo_h, hi_h = RESOLUTION_RANGE_H[priority]
        res_hours = round(rng.uniform(lo_h, hi_h) * (3 if ttype == "Feature Request" else 1), 1)
        resolved_candidate = created_at + timedelta(hours=res_hours)
        is_resolved = resolved_candidate < NOW and rng.random() < 0.95

        if is_resolved:
            status = "Resolved" if (NOW - resolved_candidate).days < 7 else "Closed"
            resolved_at = resolved_candidate
            root_cause = rng.choice(ROOT_CAUSES_BY_TYPE[ttype])
            reopen_count = rng.choices([0, 1, 2], weights=[0.82, 0.14, 0.04])[0]
        else:
            status = rng.choice(OPEN_STATUSES)
            resolved_at, res_hours, reopen_count = None, None, 0
            root_cause = "Under Investigation" if ttype in ("Bug", "Incident") else None

        is_bug = ttype == "Bug" or (ttype == "Incident" and root_cause == "Product Bug")
        escalated = priority == "P1 - Critical" or reopen_count > 0 or (priority == "P2 - High" and rng.random() < 0.3)
        support_tier = "Tier 3" if escalated else rng.choice(["Tier 1", "Tier 2"])

        # Customer feedback: only resolved/closed tickets, ~70% survey response rate
        csat = nps = sentiment = comment = None
        if is_resolved and rng.random() < 0.70:
            score = 4.6 - 1.2 * sla_breached - 0.8 * reopen_count - 0.4 * (priority == "P1 - Critical")
            score -= 0.7 * (res_hours > (lo_h + hi_h) / 2) + rng.gauss(0, 0.6)
            csat = int(min(5, max(1, round(score))))
            nps = int(min(10, max(0, round(csat * 2 + rng.gauss(0, 1.2)))))
            sentiment = "Positive" if csat >= 4 else ("Neutral" if csat == 3 else "Negative")
            if rng.random() < 0.8:
                comment = rng.choice(FEEDBACK[sentiment]).format(m=module)

        tickets.append({
            "ticket_id": f"TKT{tkt_seq:06d}",
            "account_id": acc["account_id"],
            "reporter_contact_id": rng.choice(contacts),
            "created_at": fmt_ts(created_at),
            "first_response_at": fmt_ts(first_response_at),
            "resolved_at": fmt_ts(resolved_at),
            "status": status,
            "priority": priority,
            "ticket_type": ttype,
            "product_module": module,
            "subject": subject,
            "cloud_provider": "N/A" if module == "Console & Access" else rng.choices(CLOUDS, weights=CLOUD_WEIGHTS)[0],
            "environment": rng.choices(ENVIRONMENTS, weights=ENV_WEIGHTS)[0],
            "channel": rng.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0],
            "product_version": f"2.{rng.randint(14, 19)}.{rng.randint(0, 5)}",
            "affected_assets": rng.randint(50, 5000) if priority == "P1 - Critical" else rng.randint(1, 800),
            "root_cause": root_cause,
            "is_bug": is_bug,
            "bug_id": f"UPW-{rng.randint(1000, 9999)}" if is_bug else None,
            "bug_severity": BUG_SEVERITY[priority] if is_bug else None,
            "support_tier": support_tier,
            "escalated": escalated,
            "reopen_count": reopen_count,
            "sla_first_response_minutes": sla_min,
            "first_response_minutes": first_resp_min,
            "sla_breached": sla_breached,
            "resolution_hours": res_hours,
            "csat_score": csat,
            "nps_score": nps,
            "sentiment": sentiment,
            "feedback_comment": comment,
        })
        tkt_seq += 1

print(f"Generated {len(tickets)} tickets")
tickets[:2]

## Step 4d: Extra tickets for focus accounts
Adds a varied batch of 12-18 recent Upwind support tickets for each of **Stark Labs**, **Globex Labs** and **Blue Sun Labs** (looked up by name), created between **2026-07-30 and 2026-11-01**.

* Every account gets all priorities P1-P4, including 1-2 P1 incidents, plus a mix of ticket types, modules and impact (`affected_assets`).
* Statuses are mixed: resolved/closed tickets with CSAT/NPS feedback, and open tickets (New / In Progress / Waiting on Customer).
* Tickets created after "now" (2026-10-30 09:00) are **future-dated**: they stay `New` with no first response, resolution or feedback. The same applies to any ticket whose first response would land after now (`first_response_*` empty, `sla_breached` = False).

It uses its own seeded random generator and runs after the main ticket cell, so all earlier data (accounts, transactions, employees, opportunities and the first 159 tickets) stays identical. Ticket IDs continue the `TKT` sequence.

In [0]:
FOCUS_ACCOUNT_NAMES = ["Stark Labs", "Globex Labs", "Blue Sun Labs"]
FOCUS_START = datetime(2026, 7, 30, 0, 0)
FOCUS_END = datetime(2026, 11, 1, 23, 59)
focus_rng = random.Random(SEED + 404)  # separate generator: earlier data is unaffected

acc_by_name = {a["account_name"]: a for a in accounts}
missing = [nm for nm in FOCUS_ACCOUNT_NAMES if nm not in acc_by_name]
assert not missing, f"Focus accounts not found: {missing}"

IMPACT_RANGE = {"P1 - Critical": (500, 8000), "P2 - High": (100, 2000), "P3 - Medium": (10, 500), "P4 - Low": (1, 100)}
ALL_SUBJECTS = [(m, s, t) for m in TICKET_CATALOG for (s, t) in TICKET_CATALOG[m]]

def pick_subject(priority=None, ttype=None):
    """Pick a (module, subject, type) that fits the requested priority/type."""
    if priority is None and ttype is None:
        module = focus_rng.choices(MODULES, weights=MODULE_WEIGHTS)[0]
        subject, t = focus_rng.choice(TICKET_CATALOG[module])
        return module, subject, t
    p_idx = PRIORITIES.index(priority)
    cands = [(m, s, t) for (m, s, t) in ALL_SUBJECTS
             if (ttype is None or t == ttype) and PRIORITY_WEIGHTS_BY_TYPE[t][p_idx] > 0]
    return focus_rng.choice(cands)

def random_ts(start, end):
    return start + timedelta(seconds=focus_rng.randint(0, int((end - start).total_seconds())))

focus_ticket_ids = []
for name in FOCUS_ACCOUNT_NAMES:
    acc = acc_by_name[name]
    contacts = employees_by_account[acc["account_id"]]
    n_tickets = focus_rng.randint(12, 18)

    # Guarantee variety: 1-2 P1 incidents and every priority, then random tickets
    plan = [("P1 - Critical", "Incident")] * focus_rng.randint(1, 2)
    plan += [("P2 - High", None), ("P3 - Medium", None), ("P4 - Low", None)]
    plan += [(None, None)] * (n_tickets - len(plan))
    future_slot = focus_rng.randrange(1, len(plan))  # one future-dated ticket per account (not the P1)

    for i, (forced_priority, forced_type) in enumerate(plan):
        if forced_priority:
            module, subject, ttype = pick_subject(forced_priority, forced_type)
            priority = forced_priority
        else:
            module, subject, ttype = pick_subject()
            priority = focus_rng.choices(PRIORITIES, weights=PRIORITY_WEIGHTS_BY_TYPE[ttype])[0]

        created_at = random_ts(NOW + timedelta(minutes=1), FOCUS_END) if i == future_slot else random_ts(FOCUS_START, FOCUS_END)
        sla_min = SLA_FIRST_RESPONSE_MIN[priority]
        lo, hi = FIRST_RESPONSE_RANGE_MIN[priority]
        first_resp_min = focus_rng.randint(lo, hi)
        first_response_at = created_at + timedelta(minutes=first_resp_min)

        resolved_at = res_hours = root_cause = None
        reopen_count = 0
        if created_at > NOW or first_response_at > NOW:
            # Future-dated or not yet answered: no response, resolution or feedback
            first_response_at, first_resp_min, sla_breached = None, None, False
            status = "New" if created_at > NOW else focus_rng.choice(["New", "In Progress"])
            is_resolved = False
        else:
            sla_breached = first_resp_min > sla_min
            lo_h, hi_h = RESOLUTION_RANGE_H[priority]
            res_candidate_h = round(focus_rng.uniform(lo_h, hi_h) * (3 if ttype == "Feature Request" else 1), 1)
            resolved_candidate = created_at + timedelta(hours=res_candidate_h)
            is_resolved = resolved_candidate < NOW and focus_rng.random() < 0.85
            if is_resolved:
                status = "Resolved" if (NOW - resolved_candidate).days < 7 else "Closed"
                resolved_at, res_hours = resolved_candidate, res_candidate_h
                root_cause = focus_rng.choice(ROOT_CAUSES_BY_TYPE[ttype])
                reopen_count = focus_rng.choices([0, 1, 2], weights=[0.8, 0.15, 0.05])[0]
            else:
                status = focus_rng.choice(OPEN_STATUSES)
        if not is_resolved and ttype in ("Bug", "Incident"):
            root_cause = "Under Investigation"

        is_bug = ttype == "Bug" or (ttype == "Incident" and root_cause == "Product Bug")
        escalated = priority == "P1 - Critical" or reopen_count > 0 or (priority == "P2 - High" and focus_rng.random() < 0.3)

        # Customer feedback: same CSAT/NPS logic as the main ticket cell
        csat = nps = sentiment = comment = None
        if is_resolved and focus_rng.random() < 0.75:
            lo_h, hi_h = RESOLUTION_RANGE_H[priority]
            score = 4.6 - 1.2 * sla_breached - 0.8 * reopen_count - 0.4 * (priority == "P1 - Critical")
            score -= 0.7 * (res_hours > (lo_h + hi_h) / 2) + focus_rng.gauss(0, 0.6)
            csat = int(min(5, max(1, round(score))))
            nps = int(min(10, max(0, round(csat * 2 + focus_rng.gauss(0, 1.2)))))
            sentiment = "Positive" if csat >= 4 else ("Neutral" if csat == 3 else "Negative")
            if focus_rng.random() < 0.85:
                comment = focus_rng.choice(FEEDBACK[sentiment]).format(m=module)

        lo_a, hi_a = IMPACT_RANGE[priority]
        tickets.append({
            "ticket_id": f"TKT{tkt_seq:06d}",
            "account_id": acc["account_id"],
            "reporter_contact_id": focus_rng.choice(contacts),
            "created_at": fmt_ts(created_at),
            "first_response_at": fmt_ts(first_response_at),
            "resolved_at": fmt_ts(resolved_at),
            "status": status,
            "priority": priority,
            "ticket_type": ttype,
            "product_module": module,
            "subject": subject,
            "cloud_provider": "N/A" if module == "Console & Access" else focus_rng.choices(CLOUDS, weights=CLOUD_WEIGHTS)[0],
            "environment": focus_rng.choices(ENVIRONMENTS, weights=ENV_WEIGHTS)[0],
            "channel": focus_rng.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0],
            "product_version": f"2.{focus_rng.randint(17, 20)}.{focus_rng.randint(0, 5)}",
            "affected_assets": focus_rng.randint(lo_a, hi_a),
            "root_cause": root_cause,
            "is_bug": is_bug,
            "bug_id": f"UPW-{focus_rng.randint(1000, 9999)}" if is_bug else None,
            "bug_severity": BUG_SEVERITY[priority] if is_bug else None,
            "support_tier": "Tier 3" if escalated else focus_rng.choice(["Tier 1", "Tier 2"]),
            "escalated": escalated,
            "reopen_count": reopen_count,
            "sla_first_response_minutes": sla_min,
            "first_response_minutes": first_resp_min,
            "sla_breached": sla_breached,
            "resolution_hours": res_hours,
            "csat_score": csat,
            "nps_score": nps,
            "sentiment": sentiment,
            "feedback_comment": comment,
        })
        focus_ticket_ids.append(f"TKT{tkt_seq:06d}")
        tkt_seq += 1

focus = [t for t in tickets if t["ticket_id"] in set(focus_ticket_ids)]
print(f"Added {len(focus)} focus-account tickets (total tickets: {len(tickets)})")
for name in FOCUS_ACCOUNT_NAMES:
    aid = acc_by_name[name]["account_id"]
    rows = [t for t in focus if t["account_id"] == aid]
    by_p = {p: sum(t["priority"] == p for t in rows) for p in PRIORITIES}
    print(f"{name} ({aid}): {len(rows)} tickets | " + ", ".join(f"{p.split(' ')[0]}={c}" for p, c in by_p.items()))

## Step 4e: Ticket descriptions (error details)
Adds a `description` column to every ticket, placed right after `subject`. It reads like the body of a real Upwind support case:

* **What happened:** the symptom and the error or log message for that specific subject, e.g. `CrashLoopBackOff`, `AccessDenied`, `HTTP 403`, eBPF / kprobe errors.
* **Where:** environment, Upwind version and number of affected assets (the cloud stays in its own `cloud_provider` column).
* **Current state:** the resolution and root cause for closed tickets, or that the investigation is in progress (or waiting for triage, for future-dated tickets), plus the engineering bug key (`UPW-####`) for bugs.

It uses its own seeded random generator (`SEED + 505`) and runs after all ticket cells, so every existing ticket value stays identical. Only the new column is added.

In [0]:
desc_rng = random.Random(SEED + 505)  # separate generator: existing ticket values are unaffected

# Subject -> realistic symptom / error details (one is picked per ticket)
SUBJECT_DETAILS = {
    # Runtime Sensor
    "Sensor DaemonSet crashlooping after EKS 1.30 upgrade": [
        "After upgrading the cluster to EKS 1.30 the upwind-sensor DaemonSet pods restart every 1-2 minutes with CrashLoopBackOff. Sensor log: failed to attach kprobe sys_execve: invalid argument.",
        "Since the EKS 1.30 node group rollout, sensor pods exit with code 137 (OOMKilled) right after startup and no runtime events reach the Upwind console.",
    ],
    "High CPU usage from runtime sensor on large node pool": [
        "Sensor pods consume 1.5-2 vCPU per node on a 300-node pool (expected under 0.2 vCPU), triggering HPA scale-outs and node pressure alerts.",
        "CPU throttling alerts on m6i.4xlarge nodes traced to upwind-sensor; eBPF event rate spikes above 400k events/s during batch jobs.",
    ],
    "Sensor not reporting from GKE Autopilot nodes": [
        "Nodes in the GKE Autopilot cluster show as Disconnected in Upwind for more than 24h. Sensor log: connection to collector timed out after 30s (TLS handshake).",
        "Autopilot admission rejects the sensor DaemonSet: privileged containers are not allowed, so no runtime coverage on those workloads.",
    ],
    "eBPF sensor fails to load on Bottlerocket kernel": [
        "Sensor fails at startup on Bottlerocket 1.19 nodes: BTF information not found for kernel 6.1, falling back failed - eBPF programs not loaded.",
        "On Bottlerocket nodes the sensor reports: operation not permitted while loading eBPF maps (SELinux enforcing), so the node stays without runtime protection.",
    ],
    "How to deploy the sensor with Helm behind an HTTP proxy": [
        "Customer egress goes through a corporate HTTP proxy; asking which Helm values (proxy URL, NO_PROXY) the sensor needs to reach the Upwind backend.",
        "Helm install succeeds but sensors cannot register; outbound traffic must go via proxy.corp.local:3128. Requesting the supported proxy configuration.",
    ],
    # CSPM
    "False positive: S3 bucket flagged public despite Block Public Access": [
        "Finding 'S3 bucket is publicly accessible' raised for buckets that have account-level Block Public Access enabled; the bucket policy grants access only to an internal VPC endpoint.",
        "Over 40 S3 buckets flagged Critical as public, although S3 Block Public Access is on at account level; findings reappear after manual dismissal.",
    ],
    "Misconfiguration findings not updating after remediation": [
        "Security groups were fixed via Terraform 2 days ago (0.0.0.0/0 on port 22 removed) but the findings are still Open in Upwind; last scan timestamp did not change.",
        "Remediated IAM policies still show as over-permissive; asset inventory sync for the account appears stuck at a scan from last week.",
    ],
    "Azure subscription onboarding fails with insufficient permissions": [
        "Azure onboarding wizard fails at the validation step: AuthorizationFailed - the client does not have authorization to perform action Microsoft.Resources/subscriptions/read.",
        "Onboarding of 3 new Azure subscriptions stops with 'Insufficient permissions for Reader and Security Reader roles'; the service principal was created by the customer script.",
    ],
    "Request custom policy for mandatory resource tagging": [
        "Customer wants a custom CSPM rule that flags any EC2, RDS or S3 resource missing the tags owner, cost-center and environment.",
        "Request to create an organisation-wide policy enforcing a mandatory data-classification tag on storage resources, with Critical severity for production.",
    ],
    # Vulnerability Management
    "CVE count mismatch between console and SBOM export": [
        "Console shows 1,240 open CVEs for the payments image while the SBOM / CSV export lists 1,512; the difference is mostly in OS packages.",
        "Vulnerability count on the dashboard does not match the CycloneDX SBOM export for the same image digest; customer auditors need consistent numbers.",
    ],
    "Container image scan stuck in pending state": [
        "New images pushed to ECR stay in Scan Pending for more than 6 hours; scanner log: registry pull failed - 429 Too Many Requests.",
        "Image scanning queue is stuck since the last platform update; around 300 images show Pending and CI gates that wait for scan results are blocked.",
    ],
    "Missing vulnerabilities for distroless images": [
        "Distroless and scratch-based images return 0 vulnerabilities although known CVEs exist in the bundled Go and Java libraries.",
        "Scanner does not detect language packages in gcr.io/distroless images, so critical log4j / openssl findings seen in other tools are missing in Upwind.",
    ],
    "How is runtime reachability used to prioritize CVEs?": [
        "Security team asks how Upwind decides that a CVE is reachable or in use at runtime, and how that changes the risk score shown in the console.",
        "Customer wants documentation on runtime-based CVE prioritization (loaded packages, internet exposure) to justify remediation SLAs internally.",
    ],
    "Registry scanning for private ECR repositories not running": [
        "Private ECR repositories in the secondary account are not scanned; integration status shows AccessDenied: ecr:BatchGetImage is not authorized for the Upwind role.",
        "Registry integration connected but 0 images discovered in private ECR repos in eu-west-1; cross-account role trust policy suspected.",
    ],
    # KSPM
    "CIS Kubernetes benchmark report shows stale results": [
        "CIS Kubernetes benchmark report still shows results from 10 days ago for the prod clusters, even after node pools were rebuilt.",
        "KSPM benchmark findings do not refresh: kube-bench job in the cluster completes, but results are not uploaded (HTTP 500 from the ingest endpoint).",
    ],
    "Admission controller blocking legitimate deployments": [
        "Upwind admission controller denies deployments in the payments namespace: policy 'privileged container' matched, although the pods are not privileged. Releases are blocked.",
        "After enabling enforce mode, the webhook times out (context deadline exceeded) and rejects all deployments in two clusters; customer had to set failurePolicy=Ignore.",
    ],
    "RBAC risk analysis missing ClusterRoleBindings": [
        "RBAC risk view does not list ClusterRoleBindings that grant cluster-admin to service accounts; only namespaced RoleBindings are shown.",
        "Over-privileged service accounts bound through aggregated ClusterRoles are not detected by the RBAC analysis.",
    ],
    # Cloud Detection & Response
    "Duplicate threat alerts for the same process execution": [
        "The same suspicious process execution (curl piped to sh) produces 3-5 identical alerts within a minute, flooding the SOC queue and Slack channel.",
        "CDR raises duplicate 'Reverse shell detected' alerts per container restart; deduplication window does not seem to apply.",
    ],
    "Delayed CloudTrail ingestion causing alert lag": [
        "CloudTrail-based detections arrive 2-3 hours after the event; ingestion lag metric for the org trail is above 9,000 seconds.",
        "Alerts for IAM key creation and console logins appear hours late; S3 event notifications for the CloudTrail bucket are not processed in time.",
    ],
    "Tune crypto-mining detection rule for CI runners": [
        "Crypto-mining detection fires on GitHub Actions self-hosted runners during heavy compilation jobs; customer wants a scoped exception for the CI namespace.",
        "High CPU builds on Jenkins agents are classified as crypto-mining; request to tune the rule thresholds or add an allow-list by image.",
    ],
    "Suspicious reverse shell alert - need investigation help": [
        "Critical alert: reverse shell from a production pod to an external IP over port 4444. Customer needs help reviewing the process tree and network flow to confirm or rule out compromise.",
        "Upwind detected bash spawning nc to an unknown host from the web-api container; customer requests incident-response support and evidence export.",
    ],
    "Add MITRE ATT&CK mapping to alert export": [
        "Customer SIEM requires MITRE ATT&CK tactic and technique IDs in exported alerts; current JSON export only includes Upwind rule names.",
        "Request to include ATT&CK technique IDs in webhook and CSV alert exports for SOC reporting.",
    ],
    # Compliance & Reporting
    "SOC 2 report export fails for large environments": [
        "SOC 2 compliance PDF export fails after about 5 minutes with 'Report generation failed' for the organisation with more than 40k assets.",
        "Export of the SOC 2 framework report returns HTTP 504 Gateway Timeout; smaller accounts export fine.",
    ],
    "PCI DSS framework mapping missing controls": [
        "PCI DSS v4.0 framework view is missing controls 8.3.6 and 11.6.1; auditors flagged the gap in the evidence package.",
        "Several PCI DSS requirements show Not Mapped although related CSPM rules exist and are passing.",
    ],
    "Schedule weekly compliance PDF to executives": [
        "Request to schedule a weekly compliance summary PDF (CIS and SOC 2 scores) emailed to the CISO and leadership every Monday.",
        "Customer wants automated weekly delivery of the compliance posture report instead of exporting it manually.",
    ],
    # API Security
    "API endpoint discovery missing internal services": [
        "API inventory shows only internet-facing endpoints; internal gRPC and REST services behind the service mesh are not discovered.",
        "Endpoints served by internal Kong gateway routes are missing from API discovery, although sensors run on those nodes.",
    ],
    "Shadow API detections show the wrong host": [
        "Shadow API findings list the ALB hostname instead of the actual service host header, so owners cannot be identified.",
        "Detected undocumented APIs are attributed to the ingress controller pod instead of the backend service.",
    ],
    "How to exclude health-check endpoints from API inventory": [
        "Customer asks how to exclude /healthz, /readyz and /metrics from the API inventory and risk scoring.",
        "Health-check and monitoring endpoints inflate the API count; requesting the supported way to filter them out.",
    ],
    # Integrations
    "Jira integration not creating tickets for critical findings": [
        "Jira automation rule for Critical findings stopped creating issues; integration log: 400 Bad Request - field customfield_10020 is required.",
        "No Jira tickets were created for new Critical vulnerabilities since the Jira Cloud project scheme change; integration shows Connected.",
    ],
    "Slack notifications delayed by several hours": [
        "Critical alert notifications reach the SOC Slack channel 2-4 hours late; webhook delivery log shows retries with 429 rate_limited.",
        "Slack messages for CDR alerts are delayed and arrive in bursts; customer missed an incident triage window.",
    ],
    "Splunk HEC forwarding returns 403": [
        "Splunk HEC forwarding fails: HTTP 403 Invalid token. The HEC token was rotated by the customer last week.",
        "Event forwarding to Splunk Cloud returns 403 Forbidden after the index was renamed; no Upwind events in Splunk since then.",
    ],
    "Terraform provider fails to onboard new cloud account": [
        "terraform apply with the Upwind provider fails: Error creating cloud account - 409 Conflict, account already exists, although it was deleted in the console.",
        "Upwind Terraform provider crashes during onboarding of a new AWS account: panic: runtime error: invalid memory address or nil pointer dereference.",
    ],
    "Request ServiceNow bi-directional sync": [
        "Customer wants findings to create ServiceNow incidents and ServiceNow resolution to close the finding in Upwind automatically.",
        "Request for bi-directional ServiceNow integration so that state changes sync both ways for vulnerability tickets.",
    ],
    # Console & Access
    "Dashboard widgets not loading for large organization": [
        "Main dashboard widgets spin and fail with 'Something went wrong' for the org-level view (about 60k assets); per-account views load fine.",
        "Risk overview page takes more than 60 seconds and times out in the browser for the root organisation.",
    ],
    "SSO login loop with Okta": [
        "Users are redirected back to the Okta login page in a loop after successful authentication; SAML response error: Audience restriction not met.",
        "Okta SSO login fails for all users after certificate rotation; browser shows repeated redirects between Okta and Upwind.",
    ],
    "Custom RBAC role cannot view findings": [
        "Users with the custom SecOps-Read role see an empty findings page (403 on /api/findings) although the role includes Findings: View.",
        "New custom role scoped to two cloud accounts cannot see any findings or assets; admins see the same data without issue.",
    ],
    "Add dark mode to the console": [
        "Users working in the SOC at night request a dark theme for the Upwind console.",
        "Feature request: dark mode for dashboards and alert views to reduce eye strain for on-call analysts.",
    ],
}

RESOLUTION_NOTES = {
    "Product Bug": "Root cause: product defect, fixed in a platform / sensor release.",
    "Configuration": "Root cause: customer-side configuration; fixed by correcting the setup with Upwind support.",
    "Customer Environment": "Root cause: customer environment (network, kernel or cloud permissions); resolved with a workaround.",
    "Third-party Integration": "Root cause: third-party service behaviour; resolved by adjusting the integration.",
    "Cloud Provider Issue": "Root cause: cloud provider service issue; resolved once the provider recovered.",
    "Capacity / Scaling": "Root cause: backend capacity; resolved by scaling the affected Upwind service.",
    "Documentation Gap": "Answered with documentation; docs updated to cover this case.",
    "Not a Bug": "Working as designed; behaviour explained to the customer.",
    "Feature Gap": "Logged as a product enhancement request.",
}

now_str = fmt_ts(NOW)
for t in tickets:
    details = SUBJECT_DETAILS.get(t["subject"], [f"Customer reports an issue: {t['subject']}."])
    parts = [desc_rng.choice(details)]
    # Cloud is kept in its own column (not repeated here) so the text never contradicts cloud-specific subjects
    parts.append(f"Environment: {t['environment']}, Upwind version {t['product_version']}, {t['affected_assets']} assets affected.")
    if t["priority"] == "P1 - Critical":
        parts.append(f"Business impact: critical - security coverage degraded in {t['environment'].lower()}.")
    if t["status"] in ("Resolved", "Closed"):
        parts.append(RESOLUTION_NOTES.get(t["root_cause"], "Resolved by Upwind support."))
    elif t["created_at"] > now_str:
        parts.append("Status: newly opened, waiting for triage.")
    else:
        parts.append("Status: investigation in progress.")
    if t["is_bug"]:
        parts.append(f"Tracked as engineering bug {t['bug_id']} ({t['bug_severity']}).")
    t["description"] = " ".join(parts)

if "description" not in TICKET_COLUMNS:
    TICKET_COLUMNS.insert(TICKET_COLUMNS.index("subject") + 1, "description")

missing_subjects = sorted({t["subject"] for t in tickets} - set(SUBJECT_DETAILS))
print(f"Descriptions added to {len(tickets)} tickets; subjects without specific details: {missing_subjects or 'none'}")
print(tickets[0]["description"])

## Step 5: Write the baseline CSV files (only if missing)
Each seed-42 dataset is written to its own subfolder with a header row (`accounts/accounts.csv`, `transactions/transactions.csv`, `contacts/contacts.csv`, `opportunities/opportunities.csv`, `tickets/tickets.csv`).

A baseline file is written **only if it does not exist yet**. The pipeline reads the CSVs with `allowOverwrites`, so rewriting a baseline file would append all of its rows to bronze again. Set `REGENERATE_BASE = True` to force a rewrite, for example after changing the seeded generation code. `paths` always points at the baseline files, which the Step 6 validation reads.

In [0]:
REGENERATE_BASE = False   # True = rewrite the five seed-42 baseline files even if they already exist

# entity -> (subfolder, file name, rows, columns) of the seed-42 baseline
BASE_FILES = {
    "accounts": ("accounts", "accounts.csv", accounts, ACCOUNT_COLUMNS),
    "transactions": ("transactions", "transactions.csv", transactions, TRANSACTION_COLUMNS),
    "contacts": ("contacts", "contacts.csv", employees, EMPLOYEE_COLUMNS),
    "opportunities": ("opportunities", "opportunities.csv", opportunities, OPPORTUNITY_COLUMNS),
    "tickets": ("tickets", "tickets.csv", tickets, TICKET_COLUMNS),
}

# The pipeline reads CSVs with allowOverwrites, so rewriting a baseline file re-appends all of its rows to bronze.
# Write a baseline file only when it is missing (or REGENERATE_BASE is True). paths always points at the baseline (Step 6).
paths = {}
for name, (subfolder, filename, rows, columns) in BASE_FILES.items():
    path = f"{VOLUME_PATH}/{subfolder}/{filename}"
    if REGENERATE_BASE or not os.path.exists(path):
        write_csv(subfolder, filename, rows, columns)
        action = "written"
    else:
        action = "kept (already exists)"
    paths[name] = path
    print(f"{name:<13} -> {path} ({os.path.getsize(path):,} bytes) {action}")

## Step 5b: Add a new incremental batch (every run)
Every run adds a **new batch** of data on top of the baseline. It writes one new file per entity, named with the run's UTC timestamp, and never overwrites anything:

* `accounts/accounts_<YYYYMMDDTHHMMSSZ>.csv`
* `transactions/transactions_<ts>.csv`
* `contacts/contacts_<ts>.csv`
* `opportunities/opportunities_<ts>.csv`
* `tickets/tickets_<ts>.csv`

**What a batch contains**
* 2-5 new **valid accounts** (`BATCH_ACCOUNTS`). Each gets 3-8 transactions, 2-5 contacts, 1-4 opportunities and 3-8 support tickets with descriptions. Records follow the same business rules and reference data as the seeded steps, dated relative to the **real current date**: accounts created 60 days to 3 years ago, tickets from the last 180 days up to 2 hours ago. Tickets not answered yet stay open with no first response.
* 1-2 extra **invalid account rows** (`BATCH_INVALID_ACCOUNTS`), shuffled in among the valid accounts. Each breaks 1-2 of the bronze `sf_accounts` expectations:

| Rule (bronze expectation) | How the row breaks it |
| --- | --- |
| `account_id_format` | `ACC-#####`, `acc#####`, `ACC#####X`, `ACCOUNT_#####` |
| `account_name_not_null` | empty or blank name |
| `created_date_not_null` | empty `created_date` |
| `created_date_not_future` | `created_date` 30-365 days after today |
| `account_type_allowed` | `Customr`, `Unknown`, `customer`, `Reseller` |
| `employee_count_positive` | 0 or negative |
| `annual_revenue_non_negative` | negative |
| `website_valid_format` | no scheme, no TLD, a space, or `ftp://` |

Invalid accounts get no child records. `account_id` is never empty, and every value still parses as its bronze column type. So the update never fails, and the invalid rows show up only as warnings in the pipeline Data quality tab. All other files are fully valid.

**IDs** continue from the highest ID already in each folder (`ACC#####`, `TXN######`, `EMP######`, `OPP######`, `TKT######`). The old `accounts/bad_accounts_*` test files are ignored.

**Checks before writing:** valid accounts pass all 8 rules, each invalid row breaks exactly its planned rules, and every child row is valid and references a valid account (and a contact of the same account). If any check fails, nothing is written.

This step uses its own unseeded random generator, so the seed-42 baseline is not affected.

In [0]:
import re
from datetime import datetime, timezone

BATCH_ACCOUNTS = (2, 5)          # valid new accounts per run
BATCH_INVALID_ACCOUNTS = (1, 2)  # extra invalid account rows per run (no child records)
batch_rng = random.Random()      # unseeded and separate from the seeded generators: baseline data stays identical

BATCH_NOW = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
BATCH_TODAY = BATCH_NOW.date()
BATCH_TS = BATCH_NOW.strftime("%Y%m%dT%H%M%SZ")
ALLOWED_CURRENCIES = {"USD", "EUR", "GBP", "CAD", "AUD", "JPY"}

# entity -> (folder, id column, id prefix, digits, columns)
ENTITIES = {
    "accounts": ("accounts", "account_id", "ACC", 5, ACCOUNT_COLUMNS),
    "transactions": ("transactions", "transaction_id", "TXN", 6, TRANSACTION_COLUMNS),
    "contacts": ("contacts", "contact_id", "EMP", 6, EMPLOYEE_COLUMNS),
    "opportunities": ("opportunities", "opportunity_id", "OPP", 6, OPPORTUNITY_COLUMNS),
    "tickets": ("tickets", "ticket_id", "TKT", 6, TICKET_COLUMNS),
}


def max_existing_id(folder, id_col):
    """Highest numeric part of id_col across the CSVs already in the folder (old bad_accounts_* files ignored)."""
    highest = 0
    for fn in os.listdir(f"{VOLUME_PATH}/{folder}"):
        if not fn.endswith(".csv") or fn.startswith("bad_accounts_"):
            continue
        with open(f"{VOLUME_PATH}/{folder}/{fn}", newline="") as f:
            for r in csv.DictReader(f):
                m = re.search(r"\d+", r.get(id_col) or "")
                if m:
                    highest = max(highest, int(m.group()))
    return highest


next_seq = {e: max_existing_id(folder, col) + 1 for e, (folder, col, _, _, _) in ENTITIES.items()}
start_seq = dict(next_seq)


def new_id(entity):
    _, _, prefix, digits, _ = ENTITIES[entity]
    n = next_seq[entity]
    next_seq[entity] += 1
    return f"{prefix}{n:0{digits}d}"


def b_date(start, end):
    return start + timedelta(days=batch_rng.randint(0, max((end - start).days, 0)))


def b_ts(start, end):
    return start + timedelta(seconds=batch_rng.randint(0, max(int((end - start).total_seconds()), 0)))


# ---------- valid records (same business rules as the seeded cells, dated relative to the real current date) ----------
def make_account():
    prefix, suffix = batch_rng.choice(COMPANY_PREFIXES), batch_rng.choice(COMPANY_SUFFIXES)
    emp = batch_rng.choice([batch_rng.randint(10, 200), batch_rng.randint(200, 2000), batch_rng.randint(2000, 50000)])
    domain = f"{prefix}{suffix}".lower().replace(" ", "")
    return {
        "account_id": new_id("accounts"),
        "account_name": f"{prefix} {suffix}",
        "industry": batch_rng.choice(INDUSTRIES),
        "account_type": batch_rng.choices(ACCOUNT_TYPES, weights=[0.6, 0.2, 0.2])[0],
        "country": batch_rng.choice(COUNTRIES),
        "employee_count": emp,
        "annual_revenue": round(emp * batch_rng.uniform(80_000, 400_000), -3),
        "website": f"https://www.{domain}.com",
        "created_date": b_date(BATCH_TODAY - timedelta(days=3 * 365), BATCH_TODAY - timedelta(days=60)).isoformat(),
    }


def make_transactions(acc):
    created = date.fromisoformat(acc["created_date"])
    rows = []
    for _ in range(batch_rng.randint(3, 8)):
        product = batch_rng.choice(PRODUCT_TYPES)
        low, high = AMOUNT_RANGES[product]
        rows.append({
            "transaction_id": new_id("transactions"),
            "account_id": acc["account_id"],
            "transaction_date": b_date(created + timedelta(days=1), BATCH_TODAY).isoformat(),
            "amount": round(batch_rng.uniform(low, high), 2),
            "currency": COUNTRY_CURRENCY[acc["country"]],
            "product_type": product,
            "status": batch_rng.choices(STATUSES, weights=STATUS_WEIGHTS)[0],
        })
    return rows


def make_contacts(acc):
    domain = acc["website"].replace("https://www.", "")
    created = date.fromisoformat(acc["created_date"])
    names = batch_rng.sample([(f, l) for f in FIRST_NAMES for l in LAST_NAMES], batch_rng.randint(2, 5))
    rows = []
    for first, last in names:
        dept = batch_rng.choice(list(TITLES_BY_DEPT))
        rows.append({
            "contact_id": new_id("contacts"),
            "account_id": acc["account_id"],
            "first_name": first,
            "last_name": last,
            "email": f"{first.lower()}.{last.lower()}@{domain}",
            "title": batch_rng.choice(TITLES_BY_DEPT[dept]),
            "department": dept,
            "phone": f"+1-{batch_rng.randint(200, 999)}-{batch_rng.randint(200, 999)}-{batch_rng.randint(1000, 9999)}",
            "start_date": b_date(created, BATCH_TODAY).isoformat(),
        })
    return rows


def make_opportunities(acc, contact_ids):
    created = date.fromisoformat(acc["created_date"])
    rows = []
    for _ in range(batch_rng.randint(1, 4)):
        stage = batch_rng.choices(STAGES, weights=STAGE_WEIGHTS)[0]
        is_closed = stage.startswith("Closed")
        opp_created = b_date(created, BATCH_TODAY - timedelta(days=30))
        if is_closed:
            close = b_date(opp_created + timedelta(days=14), BATCH_TODAY - timedelta(days=1))
        else:
            close = b_date(BATCH_TODAY + timedelta(days=1), BATCH_TODAY + timedelta(days=270))
        opp_type = batch_rng.choice(OPP_TYPES)
        rows.append({
            "opportunity_id": new_id("opportunities"),
            "account_id": acc["account_id"],
            "opportunity_name": f"{acc['account_name']} - {batch_rng.choice(OPP_PRODUCTS)} ({opp_type})",
            "type": opp_type,
            "stage": stage,
            "probability": STAGE_PROBABILITY[stage],
            "amount": round(batch_rng.uniform(10_000, 500_000), -2),
            "currency": COUNTRY_CURRENCY[acc["country"]],
            "lead_source": batch_rng.choice(LEAD_SOURCES),
            "primary_contact_id": batch_rng.choice(contact_ids),
            "created_date": opp_created.isoformat(),
            "close_date": close.isoformat(),
            "is_closed": is_closed,
            "is_won": stage == "Closed Won",
        })
    return rows


def describe_ticket(t):
    """Same description format as Step 4e."""
    details = SUBJECT_DETAILS.get(t["subject"], [f"Customer reports an issue: {t['subject']}."])
    parts = [batch_rng.choice(details),
             f"Environment: {t['environment']}, Upwind version {t['product_version']}, {t['affected_assets']} assets affected."]
    if t["priority"] == "P1 - Critical":
        parts.append(f"Business impact: critical - security coverage degraded in {t['environment'].lower()}.")
    if t["status"] in ("Resolved", "Closed"):
        parts.append(RESOLUTION_NOTES.get(t["root_cause"], "Resolved by Upwind support."))
    elif t["first_response_at"] is None:
        parts.append("Status: newly opened, waiting for triage.")
    else:
        parts.append("Status: investigation in progress.")
    if t["is_bug"]:
        parts.append(f"Tracked as engineering bug {t['bug_id']} ({t['bug_severity']}).")
    return " ".join(parts)


def make_tickets(acc, contact_ids):
    window_start = max(datetime.fromisoformat(acc["created_date"]), BATCH_NOW - timedelta(days=180))
    window_end = BATCH_NOW - timedelta(hours=2)
    rows = []
    for _ in range(batch_rng.randint(3, 8)):
        module = batch_rng.choices(MODULES, weights=MODULE_WEIGHTS)[0]
        subject, ttype = batch_rng.choice(TICKET_CATALOG[module])
        priority = batch_rng.choices(PRIORITIES, weights=PRIORITY_WEIGHTS_BY_TYPE[ttype])[0]
        created_at = b_ts(window_start, window_end)

        sla_min = SLA_FIRST_RESPONSE_MIN[priority]
        lo, hi = FIRST_RESPONSE_RANGE_MIN[priority]
        first_resp_min = batch_rng.randint(lo, hi)
        first_response_at = created_at + timedelta(minutes=first_resp_min)
        answered = first_response_at <= BATCH_NOW
        if not answered:   # first response would be in the future: not answered yet
            first_resp_min, first_response_at = None, None
        sla_breached = answered and first_resp_min > sla_min

        lo_h, hi_h = RESOLUTION_RANGE_H[priority]
        res_hours = round(batch_rng.uniform(lo_h, hi_h) * (3 if ttype == "Feature Request" else 1), 1)
        resolved_candidate = created_at + timedelta(hours=res_hours)
        is_resolved = answered and resolved_candidate < BATCH_NOW and batch_rng.random() < 0.95

        if is_resolved:
            status = "Resolved" if (BATCH_NOW - resolved_candidate).days < 7 else "Closed"
            resolved_at = resolved_candidate
            root_cause = batch_rng.choice(ROOT_CAUSES_BY_TYPE[ttype])
            reopen_count = batch_rng.choices([0, 1, 2], weights=[0.82, 0.14, 0.04])[0]
        else:
            status = batch_rng.choice(OPEN_STATUSES) if answered else "New"
            resolved_at, res_hours, reopen_count = None, None, 0
            root_cause = "Under Investigation" if ttype in ("Bug", "Incident") else None

        is_bug = ttype == "Bug" or (ttype == "Incident" and root_cause == "Product Bug")
        escalated = priority == "P1 - Critical" or reopen_count > 0 or (priority == "P2 - High" and batch_rng.random() < 0.3)

        csat = nps = sentiment = comment = None
        if is_resolved and batch_rng.random() < 0.70:
            score = 4.6 - 1.2 * sla_breached - 0.8 * reopen_count - 0.4 * (priority == "P1 - Critical")
            score -= 0.7 * (res_hours > (lo_h + hi_h) / 2) + batch_rng.gauss(0, 0.6)
            csat = int(min(5, max(1, round(score))))
            nps = int(min(10, max(0, round(csat * 2 + batch_rng.gauss(0, 1.2)))))
            sentiment = "Positive" if csat >= 4 else ("Neutral" if csat == 3 else "Negative")
            if batch_rng.random() < 0.8:
                comment = batch_rng.choice(FEEDBACK[sentiment]).format(m=module)

        t = {
            "ticket_id": new_id("tickets"),
            "account_id": acc["account_id"],
            "reporter_contact_id": batch_rng.choice(contact_ids),
            "created_at": fmt_ts(created_at),
            "first_response_at": fmt_ts(first_response_at),
            "resolved_at": fmt_ts(resolved_at),
            "status": status,
            "priority": priority,
            "ticket_type": ttype,
            "product_module": module,
            "subject": subject,
            "cloud_provider": "N/A" if module == "Console & Access" else batch_rng.choices(CLOUDS, weights=CLOUD_WEIGHTS)[0],
            "environment": batch_rng.choices(ENVIRONMENTS, weights=ENV_WEIGHTS)[0],
            "channel": batch_rng.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0],
            "product_version": f"2.{batch_rng.randint(14, 19)}.{batch_rng.randint(0, 5)}",
            "affected_assets": batch_rng.randint(50, 5000) if priority == "P1 - Critical" else batch_rng.randint(1, 800),
            "root_cause": root_cause,
            "is_bug": is_bug,
            "bug_id": f"UPW-{batch_rng.randint(1000, 9999)}" if is_bug else None,
            "bug_severity": BUG_SEVERITY[priority] if is_bug else None,
            "support_tier": "Tier 3" if escalated else batch_rng.choice(["Tier 1", "Tier 2"]),
            "escalated": escalated,
            "reopen_count": reopen_count,
            "sla_first_response_minutes": sla_min,
            "first_response_minutes": first_resp_min,
            "sla_breached": sla_breached,
            "resolution_hours": res_hours,
            "csat_score": csat,
            "nps_score": nps,
            "sentiment": sentiment,
            "feedback_comment": comment,
        }
        t["description"] = describe_ticket(t)
        rows.append(t)
    return rows


# ---------- invalid accounts: each breaks 1-2 bronze sf_accounts expectations ----------
def _bad_account_id(row):
    n = re.search(r"\d+", row["account_id"]).group()
    row["account_id"] = batch_rng.choice([f"ACC-{n}", f"acc{n}", f"ACC{n}X", f"ACCOUNT_{n}"])

def _bad_name(row):
    row["account_name"] = batch_rng.choice(["", "   "])

def _null_created(row):
    row["created_date"] = ""

def _future_created(row):
    row["created_date"] = (BATCH_TODAY + timedelta(days=batch_rng.randint(30, 365))).isoformat()

def _bad_type(row):
    row["account_type"] = batch_rng.choice(["Customr", "Unknown", "customer", "Reseller"])

def _bad_employees(row):
    row["employee_count"] = batch_rng.choice([0, -batch_rng.randint(1, 500)])

def _bad_revenue(row):
    row["annual_revenue"] = -round(batch_rng.uniform(1_000, 10_000_000), -3)

def _bad_website(row):
    domain = row["website"].replace("https://www.", "").replace(".com", "")
    row["website"] = batch_rng.choice([f"www.{domain}.com", f"https://{domain}", f"https://{domain} site.com", f"ftp://{domain}.com"])

BREAK_RULE = {
    "account_id_format": _bad_account_id,
    "account_name_not_null": _bad_name,
    "created_date_not_null": _null_created,
    "created_date_not_future": _future_created,
    "account_type_allowed": _bad_type,
    "employee_count_positive": _bad_employees,
    "annual_revenue_non_negative": _bad_revenue,
    "website_valid_format": _bad_website,
}
DATE_RULES = {"created_date_not_null", "created_date_not_future"}   # mutually exclusive on one row

# Python versions of the bronze sf_accounts expectations (True = row passes).
# A NULL created_date is treated as passing created_date_not_future here; it is caught by created_date_not_null.
ACCOUNT_RULES = {
    "account_id_format": lambda r: re.fullmatch(r"ACC[0-9]+", r["account_id"]) is not None,
    "account_name_not_null": lambda r: r["account_name"].strip() != "",
    "created_date_not_null": lambda r: r["created_date"] != "",
    "created_date_not_future": lambda r: r["created_date"] == "" or date.fromisoformat(r["created_date"]) <= BATCH_TODAY,
    "account_type_allowed": lambda r: r["account_type"] in ACCOUNT_TYPES,
    "employee_count_positive": lambda r: int(r["employee_count"]) > 0,
    "annual_revenue_non_negative": lambda r: float(r["annual_revenue"]) >= 0,
    "website_valid_format": lambda r: re.fullmatch(r"https?://[^ ]+\.[a-z]{2,}", r["website"]) is not None,
}


def failed_account_rules(row):
    return {name for name, ok in ACCOUNT_RULES.items() if not ok(row)}


# ---------- build the batch ----------
batch = {e: [] for e in ENTITIES}
for _ in range(batch_rng.randint(*BATCH_ACCOUNTS)):
    acc = make_account()
    acc_contacts = make_contacts(acc)
    contact_ids = [c["contact_id"] for c in acc_contacts]
    batch["accounts"].append(acc)
    batch["contacts"] += acc_contacts
    batch["transactions"] += make_transactions(acc)
    batch["opportunities"] += make_opportunities(acc, contact_ids)
    batch["tickets"] += make_tickets(acc, contact_ids)
valid_account_ids = {a["account_id"] for a in batch["accounts"]}

invalid_log = []
for _ in range(batch_rng.randint(*BATCH_INVALID_ACCOUNTS)):
    row = make_account()
    planned = batch_rng.sample(list(BREAK_RULE), batch_rng.randint(1, 2))
    if DATE_RULES <= set(planned):
        planned.remove("created_date_not_null")
    for rule in planned:
        BREAK_RULE[rule](row)
    batch["accounts"].append(row)
    invalid_log.append((row, set(planned)))
batch_rng.shuffle(batch["accounts"])


# ---------- verify before writing ----------
def as_csv_row(r):
    """Values as the CSV / bronze reader sees them (None -> empty string)."""
    return {k: "" if v is None else str(v) for k, v in r.items()}

problems = []
now_s = fmt_ts(BATCH_NOW)

for a in batch["accounts"]:
    r = as_csv_row(a)
    if not r["account_id"]:
        problems.append("account with empty account_id")
    int(r["employee_count"]); float(r["annual_revenue"])              # must parse as INT / DOUBLE
    if r["created_date"]:
        date.fromisoformat(r["created_date"])                          # must parse as DATE
    if a["account_id"] in valid_account_ids and failed_account_rules(r):
        problems.append(f"valid account {a['account_id']} fails {failed_account_rules(r)}")
for row, planned in invalid_log:
    got = failed_account_rules(as_csv_row(row))
    if got != planned:
        problems.append(f"invalid account {row['account_id']}: planned {planned}, breaks {got}")

contact_account = {c["contact_id"]: c["account_id"] for c in batch["contacts"]}
acc_created = {a["account_id"]: a["created_date"] for a in batch["accounts"] if a["account_id"] in valid_account_ids}
for t in batch["transactions"]:
    ok = (re.fullmatch(r"TXN[0-9]+", t["transaction_id"]) and t["account_id"] in valid_account_ids
          and acc_created[t["account_id"]] < t["transaction_date"] <= BATCH_TODAY.isoformat() and t["amount"] > 0
          and t["currency"] in ALLOWED_CURRENCIES and t["product_type"] in PRODUCT_TYPES and t["status"] in STATUSES)
    if not ok: problems.append(f"invalid transaction {t['transaction_id']}")
for c in batch["contacts"]:
    ok = (re.fullmatch(r"EMP[0-9]+", c["contact_id"]) and c["account_id"] in valid_account_ids and c["last_name"].strip()
          and re.fullmatch(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", c["email"])
          and acc_created[c["account_id"]] <= c["start_date"] <= BATCH_TODAY.isoformat())
    if not ok: problems.append(f"invalid contact {c['contact_id']}")
for o in batch["opportunities"]:
    ok = (re.fullmatch(r"OPP[0-9]+", o["opportunity_id"]) and o["account_id"] in valid_account_ids
          and contact_account.get(o["primary_contact_id"]) == o["account_id"] and o["stage"] in STAGE_PROBABILITY
          and o["type"] in OPP_TYPES and o["currency"] in ALLOWED_CURRENCIES and 0 <= o["probability"] <= 100 and o["amount"] > 0
          and o["is_closed"] == o["stage"].startswith("Closed") and o["is_won"] == (o["stage"] == "Closed Won")
          and acc_created[o["account_id"]] <= o["created_date"] <= o["close_date"]
          and (o["close_date"] < BATCH_TODAY.isoformat()) == o["is_closed"])
    if not ok: problems.append(f"invalid opportunity {o['opportunity_id']}")
RESOLVED = {"Resolved", "Closed"}
for t in batch["tickets"]:
    fr, res = t["first_response_at"], t["resolved_at"]
    ok = (re.fullmatch(r"TKT[0-9]+", t["ticket_id"]) and t["account_id"] in valid_account_ids
          and contact_account.get(t["reporter_contact_id"]) == t["account_id"]
          and acc_created[t["account_id"]] <= t["created_at"][:10] and t["created_at"] <= now_s
          and len(t["description"].strip()) >= 20 and t["status"] in OPEN_STATUSES + ["Resolved", "Closed"]
          and t["priority"] in PRIORITIES and t["product_module"] in TICKET_CATALOG
          and t["ticket_type"] in PRIORITY_WEIGHTS_BY_TYPE
          and (fr is None or t["created_at"] <= fr <= now_s) and (res is None or res >= t["created_at"])
          and ((res is not None) == (t["status"] in RESOLVED)) and (fr is not None or t["status"] not in RESOLVED)
          and (t["first_response_minutes"] is None or t["sla_breached"] == (t["first_response_minutes"] > t["sla_first_response_minutes"]))
          and (t["first_response_minutes"] is not None or not t["sla_breached"])
          and (t["csat_score"] is None or (t["status"] in RESOLVED and 1 <= t["csat_score"] <= 5))
          and (t["nps_score"] is None or 0 <= t["nps_score"] <= 10)
          and ((t["bug_id"] is not None) == t["is_bug"]) and t["reopen_count"] >= 0)
    if not ok: problems.append(f"invalid ticket {t['ticket_id']}")
for e, rows in batch.items():
    ids = [r[ENTITIES[e][1]] for r in rows]
    if len(ids) != len(set(ids)):
        problems.append(f"duplicate {e} ids in batch")

assert not problems, "Batch checks failed, nothing written:\n" + "\n".join(problems)


# ---------- write one new file per entity ----------
batch_paths = {}
for e, (folder, _, _, _, columns) in ENTITIES.items():
    fn = f"{e}_{BATCH_TS}.csv"
    assert not os.path.exists(f"{VOLUME_PATH}/{folder}/{fn}"), f"{fn} already exists"
    batch_paths[e] = write_csv(folder, fn, batch[e], columns)

print(f"Batch {BATCH_TS}: all checks passed ({len(valid_account_ids)} valid + {len(invalid_log)} invalid accounts)")
for e, p in batch_paths.items():
    print(f"  {e:<13} {len(batch[e]):>3} rows -> {p}   (IDs from {start_seq[e]})")
print("Invalid account rows:")
for row, planned in invalid_log:
    print(f"  {row['account_id']!r:<16} breaks: {', '.join(sorted(planned))}")

## Step 6: Validate the output
Re-read the files from the volume and check that:
* all 3 files exist and row counts are within expected ranges
* every account has at least one transaction and one employee
* there are no orphan `account_id`s in transactions or employees
* every transaction date is after its account's created date

In [0]:
def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

for p in paths.values():
    assert os.path.exists(p), f"Missing file: {p}"

acc_rows = read_csv(paths["accounts"])
txn_rows = read_csv(paths["transactions"])
emp_rows = read_csv(paths["contacts"])
opp_rows = read_csv(paths["opportunities"])
tkt_rows = read_csv(paths["tickets"])
n = len(acc_rows)

acc_ids = {r["account_id"] for r in acc_rows}
txn_acc_ids = {r["account_id"] for r in txn_rows}
emp_acc_ids = {r["account_id"] for r in emp_rows}
acc_created = {r["account_id"]: r["created_date"] for r in acc_rows}

assert len(acc_ids) == n, "Duplicate account_ids"
assert 3 * n <= len(txn_rows) <= 8 * n, f"Unexpected transaction count {len(txn_rows)}"
assert 2 * n <= len(emp_rows) <= 5 * n, f"Unexpected employee count {len(emp_rows)}"
assert acc_ids == txn_acc_ids, f"Accounts without transactions: {acc_ids - txn_acc_ids}"
assert acc_ids == emp_acc_ids, f"Accounts without employees: {acc_ids - emp_acc_ids}"
assert txn_acc_ids <= acc_ids, f"Orphan transaction account_ids: {txn_acc_ids - acc_ids}"
assert emp_acc_ids <= acc_ids, f"Orphan employee account_ids: {emp_acc_ids - acc_ids}"
assert all(r["transaction_date"] > acc_created[r["account_id"]] for r in txn_rows), "Transaction before account creation"

# Opportunities
emp_account = {r["contact_id"]: r["account_id"] for r in emp_rows}
opp_acc_ids = {r["account_id"] for r in opp_rows}
assert n <= len(opp_rows) <= 4 * n, f"Unexpected opportunity count {len(opp_rows)}"
assert len({r["opportunity_id"] for r in opp_rows}) == len(opp_rows), "Duplicate opportunity_ids"
assert acc_ids == opp_acc_ids, f"Accounts without opportunities: {acc_ids - opp_acc_ids}"
assert all(emp_account.get(r["primary_contact_id"]) == r["account_id"] for r in opp_rows), "Primary contact not on the same account"
assert all(r["created_date"] >= acc_created[r["account_id"]] for r in opp_rows), "Opportunity before account creation"
assert all((r["close_date"] < TODAY.isoformat()) == (r["is_closed"] == "True") for r in opp_rows), "close_date inconsistent with stage"

# Tickets
RESOLVED = {"Resolved", "Closed"}
assert 3 * n <= len(tkt_rows) <= 12 * n + 60, f"Unexpected ticket count {len(tkt_rows)}"
assert len({r["ticket_id"] for r in tkt_rows}) == len(tkt_rows), "Duplicate ticket_ids"
assert acc_ids == {r["account_id"] for r in tkt_rows}, "Accounts without tickets"
assert all(emp_account.get(r["reporter_contact_id"]) == r["account_id"] for r in tkt_rows), "Reporter not on the same account"
assert all(r["created_at"][:10] >= acc_created[r["account_id"]] for r in tkt_rows), "Ticket before account creation"
assert all(r["first_response_at"] == "" or r["first_response_at"] >= r["created_at"] for r in tkt_rows), "First response before creation"
assert all(r["first_response_at"] != "" or (r["status"] not in RESOLVED and r["sla_breached"] == "False") for r in tkt_rows), "Unanswered ticket must be open and not SLA-breached"
now_str = fmt_ts(NOW)
future_rows = [r for r in tkt_rows if r["created_at"] > now_str]
assert all(r["status"] not in RESOLVED and r["first_response_at"] == "" and r["csat_score"] == "" and r["feedback_comment"] == "" for r in future_rows), "Future-dated tickets must be open with no response/feedback"
for nm in FOCUS_ACCOUNT_NAMES:
    aid = acc_by_name[nm]["account_id"]
    recent = [r for r in tkt_rows if r["account_id"] == aid and r["created_at"] >= "2026-07-30"]
    assert {r["priority"] for r in recent} == set(PRIORITIES), f"{nm}: not all priorities present"
assert all((r["resolved_at"] != "") == (r["status"] in RESOLVED) for r in tkt_rows), "resolved_at inconsistent with status"
assert all(r["resolved_at"] == "" or r["resolved_at"] >= r["created_at"] for r in tkt_rows), "Resolved before creation"
assert all(r["csat_score"] == "" or (r["status"] in RESOLVED and 1 <= int(r["csat_score"]) <= 5) for r in tkt_rows), "Invalid CSAT"
assert all((r["bug_id"] != "") == (r["is_bug"] == "True") for r in tkt_rows), "bug_id inconsistent with is_bug"
assert all(len(r["description"]) >= 40 for r in tkt_rows), "Ticket without a description"
assert all(r["bug_id"] == "" or r["bug_id"] in r["description"] for r in tkt_rows), "Bug description missing its bug_id"

print(f"accounts:     {n} rows")
print(f"transactions: {len(txn_rows)} rows (expected {3*n}-{8*n})")
print(f"employees:    {len(emp_rows)} rows (expected {2*n}-{5*n})")
print(f"opportunities: {len(opp_rows)} rows (expected {n}-{4*n})")
print(f"tickets:      {len(tkt_rows)} rows (expected {3*n}-{12*n + 60}), future-dated: {len(future_rows)}")
print("All validation checks passed.")